In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import os

DATASET_PATH = os.path.join(os.getcwd(),'data','20230601_prof.nc')

data = xr.open_dataset(DATASET_PATH)
data

<xarray.Dataset> Size: 6MB
Dimensions:                       (N_PROF: 55, N_PARAM: 3, N_LEVELS: 1242,
                                   N_CALIB: 2, N_HISTORY: 0)
Dimensions without coordinates: N_PROF, N_PARAM, N_LEVELS, N_CALIB, N_HISTORY
Data variables: (12/64)
    DATA_TYPE                     object 8B ...
    FORMAT_VERSION                object 8B ...
    HANDBOOK_VERSION              object 8B ...
    REFERENCE_DATE_TIME           object 8B ...
    DATE_CREATION                 object 8B ...
    DATE_UPDATE                   object 8B ...
    ...                            ...
    HISTORY_ACTION                (N_HISTORY, N_PROF) object 0B ...
    HISTORY_PARAMETER             (N_HISTORY, N_PROF) object 0B ...
    HISTORY_START_PRES            (N_HISTORY, N_PROF) float32 0B ...
    HISTORY_STOP_PRES             (N_HISTORY, N_PROF) float32 0B ...
    HISTORY_PREVIOUS_VALUE        (N_HISTORY, N_PROF) float32 0B ...
    HISTORY_QCTEST                (N_HISTORY, N_PROF) object 0B ...
Attributes:
    title:                Argo float vertical profile
    institution:          FR GDAC
    source:               Argo float
    history:              2026-06-03T06:53:33Z creation
    references:           http://www.argodatamgt.org/Documentation
    user_manual_version:  3.1
    Conventions:          Argo-3.1 CF-1.6
    featureType:          trajectoryProfile

In [6]:
list(data.data_vars)

['DATA_TYPE',
 'FORMAT_VERSION',
 'HANDBOOK_VERSION',
 'REFERENCE_DATE_TIME',
 'DATE_CREATION',
 'DATE_UPDATE',
 'PLATFORM_NUMBER',
 'PROJECT_NAME',
 'PI_NAME',
 'STATION_PARAMETERS',
 'CYCLE_NUMBER',
 'DIRECTION',
 'DATA_CENTRE',
 'DC_REFERENCE',
 'DATA_STATE_INDICATOR',
 'DATA_MODE',
 'PLATFORM_TYPE',
 'FLOAT_SERIAL_NO',
 'FIRMWARE_VERSION',
 'WMO_INST_TYPE',
 'JULD',
 'JULD_QC',
 'JULD_LOCATION',
 'LATITUDE',
 'LONGITUDE',
 'POSITION_QC',
 'POSITIONING_SYSTEM',
 'PROFILE_PRES_QC',
 'PROFILE_TEMP_QC',
 'PROFILE_PSAL_QC',
 'VERTICAL_SAMPLING_SCHEME',
 'CONFIG_MISSION_NUMBER',
 'PRES',
 'PRES_QC',
 'PRES_ADJUSTED',
 'PRES_ADJUSTED_QC',
 'PRES_ADJUSTED_ERROR',
 'TEMP',
 'TEMP_QC',
 'TEMP_ADJUSTED',
 'TEMP_ADJUSTED_QC',
 'TEMP_ADJUSTED_ERROR',
 'PSAL',
 'PSAL_QC',
 'PSAL_ADJUSTED',
 'PSAL_ADJUSTED_QC',
 'PSAL_ADJUSTED_ERROR',
 'PARAMETER',
 'SCIENTIFIC_CALIB_EQUATION',
 'SCIENTIFIC_CALIB_COEFFICIENT',
 'SCIENTIFIC_CALIB_COMMENT',
 'SCIENTIFIC_CALIB_DATE',
 'HISTORY_INSTITUTION',
 'HISTOR

In [7]:
essential_vars = ['PLATFORM_NUMBER', 'LATITUDE', 'LONGITUDE', 'JULD', 'PRES', 'TEMP', 'PSAL']

df = data[essential_vars].to_dataframe().reset_index()
df

,N_PROF,N_LEVELS,PLATFORM_NUMBER,LATITUDE,LONGITUDE,JULD,PRES,TEMP,PSAL
0,0,0,b'1902482 ',0.98644,65.20156,2023-06-01 23:56:49,1.12,30.127001,34.835999
1,0,1,b'1902482 ',0.98644,65.20156,2023-06-01 23:56:49,2.00,30.132999,34.834000
2,0,2,b'1902482 ',0.98644,65.20156,2023-06-01 23:56:49,3.00,30.135000,34.833000
3,0,3,b'1902482 ',0.98644,65.20156,2023-06-01 23:56:49,4.00,30.129000,34.831001
4,0,4,b'1902482 ',0.98644,65.20156,2023-06-01 23:56:49,4.96,30.136000,34.830002
...,...,...,...,...,...,...,...,...,...
68305,54,1237,b'1902473 ',-2.98064,63.81826,2023-06-01 01:15:38,NaN,NaN,NaN
68306,54,1238,b'1902473 ',-2.98064,63.81826,2023-06-01 01:15:38,NaN,NaN,NaN
68307,54,1239,b'1902473 ',-2.98064,63.81826,2023-06-01 01:15:38,NaN,NaN,NaN
68308,54,1240,b'1902473 ',-2.98064,63.81826,2023-06-01 01:15:38,NaN,NaN,NaN


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 68310 entries, 0 to 68309
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   N_PROF           68310 non-null  int64         
 1   N_LEVELS         68310 non-null  int64         
 2   PLATFORM_NUMBER  68310 non-null  object        
 3   LATITUDE         68310 non-null  float64       
 4   LONGITUDE        68310 non-null  float64       
 5   JULD             68310 non-null  datetime64[ns]
 6   PRES             35562 non-null  float32       
 7   TEMP             35562 non-null  float32       
 8   PSAL             35562 non-null  float32       
dtypes: datetime64[ns](1), float32(3), float64(2), int64(2), object(1)
memory usage: 3.9+ MB


In [9]:
cleaned_df = df.dropna()

In [10]:
import sqlite3

conn = sqlite3.connect('data/argo_data.db')

#Table name :"measurements" conn, if_exists="replace", index=False
cleaned_df.to_sql(name='measurements',con=conn,if_exists='replace',index=False)

print('Rows added to DB')

Rows added to DB


In [11]:
query = '''
    SELECT 
        Platform_number,
        COUNT(*) as total_readings,
        ROUND(MIN(temp), 2) as min_temp,
        ROUND(MAX(temp), 2) as max_temp,
        ROUND(AVG(latitude), 4) as avg_lat,
        ROUND(AVG(longitude), 4) as avg_lon
    FROM measurements
    GROUP BY Platform_number
'''

cursor = conn.cursor()
cursor.execute(query)

rows = cursor.fetchall()

for row in rows :
    print(f"Float #{row[0].decode().strip()} took {row[1]} readings. Temp range: {row[2]}°C to {row[3]}°C. Location: ({row[4]}, {row[5]})")

Float #1901743 took 995 readings. Temp range: 0.59°C to 1.97°C. Location: (-54.6967, 79.1357)
Float #1901809 took 1012 readings. Temp range: 2.46°C to 12.54°C. Location: (-40.2299, 121.9377)
Float #1901843 took 1010 readings. Temp range: 2.4°C to 27.43°C. Location: (-14.7126, 68.8273)
Float #1901897 took 54 readings. Temp range: 3.22°C to 30.43°C. Location: (16.7665, 55.1972)
Float #1902043 took 1012 readings. Temp range: 2.76°C to 19.37°C. Location: (-35.145, 47.9535)
Float #1902222 took 1010 readings. Temp range: 2.65°C to 16.44°C. Location: (-38.4063, 69.8896)
Float #1902259 took 1012 readings. Temp range: 2.57°C to 17.54°C. Location: (-36.2464, 62.9334)
Float #1902292 took 1008 readings. Temp range: 0.61°C to 1.84°C. Location: (-55.9838, 48.396)
Float #1902294 took 1012 readings. Temp range: 0.43°C to 1.84°C. Location: (-57.3648, 34.1121)
Float #1902453 took 510 readings. Temp range: 2.81°C to 30.05°C. Location: (-0.9638, 64.6365)
Float #1902472 took 501 readings. Temp range: 7.67°

In [14]:
import sqlite3

# 1. Connect to your database
conn = sqlite3.connect('data/argo_data.db')
cursor = conn.cursor()

# 2. Query to list and count unique floats inside the Arabian Sea boundary
verification_query = '''
    SELECT COUNT(DISTINCT Platform_number) FROM measurements WHERE latitude BETWEEN 8 AND 26 AND longitude BETWEEN 32 AND 80
'''

cursor.execute(verification_query)
rows = cursor.fetchall()

print(f"--- Verification Results: Found {len(rows)} Unique Floats ---")
for row in rows:
    print(row)

conn.close()

--- Verification Results: Found 1 Unique Floats ---
(4,)
